In [ ]:
import pandas as pd
import numpy as np
import arviz as az
import numpyro
from jax import numpy as jnp
from jax.random import PRNGKey
from numpyro import distributions as dist
from numpyro import infer
from plotly import graph_objects as go

from summer3.epi import (
    CompartmentalModelODE,
    CategoryData,
    strat_data_from_pandas,
    build_istate,
    dti_to_epoch,
)

from tb_macro.constants import ALL_COMPARTMENTS, AGE_STRATA, MAX_AGE
from tb_macro.epi import (
    get_base_model,
    add_natural_history,
    add_seeding,
    add_latency_flows,
    add_infection_flows,
)
from tb_macro.inputs import (
    get_country_pop,
    get_single_age_pop_from_ungroups,
    get_group_popsizes,
    get_un_mortality,
    load_conmat,
    convert_conmat,
    normalise_spectral_radius,
    add_groups_to_single_pop,
    build_age_weight_lookup,
    get_fertility_data,
)
from tb_macro.demography import (
    add_replacement_deaths,
    add_ageing_flows,
    prepare_pop_data_for_entries,
    add_entry_births,
)
from tb_macro.health_system import add_treatment_flows, add_detection

pd.options.plotting.backend = "plotly"

In [ ]:
iso3 = "VNM"
start_time = 1800.0
end_time = 2030.0
young_end_age = 15

In [ ]:
# Data loading and processing
pop_data = get_country_pop(iso3)
single_age_pops = get_single_age_pop_from_ungroups(pop_data)
group_popsize = get_group_popsizes(single_age_pops)
agg_pop_data = group_popsize.sum(axis=1)
mort_data = get_un_mortality(iso3)
death_rates = mort_data.div(group_popsize, axis=0).dropna()
conmat_data = load_conmat(iso3)
matrix = convert_conmat(conmat_data)
norm_matrix = normalise_spectral_radius(matrix)
add_groups_to_single_pop(single_age_pops)
age_weights = build_age_weight_lookup(single_age_pops)
fert = get_fertility_data(iso3)
fert_padded = fert.reindex(columns=range(MAX_AGE + 1), fill_value=0.0)

In [ ]:
# Model construction
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model(
    start_time, end_time
)
start_pops = [1000.0] * len(AGE_STRATA)
times, rates = prepare_pop_data_for_entries(agg_pop_data, start_time, sum(start_pops))

add_infection_flows(
    epi_model,
    disease_state,
    age_strat,
    clin_strat,
    infect_strat,
    age_weights,
    group_popsize,
    fert_padded,
    young_end_age,
)
add_natural_history(epi_model, disease_state, clin_strat, infect_strat)
add_ageing_flows(epi_model, age_strat)
add_seeding(epi_model, disease_state)
add_detection(epi_model, disease_state, clin_strat)
add_replacement_deaths(epi_model, disease_state, age_strat, death_rates, start_time)
add_entry_births(epi_model, disease_state, age_strat, start_time, rates, times)
add_treatment_flows(
    death_rates,
    start_time,
    epi_model,
    disease_state,
    age_strat,
    infect_strat,
    clin_strat,
)
add_latency_flows(epi_model, disease_state, age_strat, clin_strat, infect_strat)

# Initialisation
data = pd.Series(index=[str(a) for a in AGE_STRATA], data=np.array(start_pops))
base_pops = strat_data_from_pandas(data, age_strat)
init_splits = [0.0] * len(ALL_COMPARTMENTS)
init_splits[ALL_COMPARTMENTS.index("mtb_naive")] = 1.0
pop_splits = [CategoryData(disease_state.categories(), jnp.array((init_splits)))]
epi_model.set_initial_population(base_pops, pop_splits)

In [ ]:
def get_runner(epi_model):
    istate = build_istate(epi_model.cmap, epi_model.base_pops, epi_model.pop_splits)
    cmodel = CompartmentalModelODE(epi_model.cmap, epi_model.flows)
    runner = cmodel.get_runner(
        len(epi_model.times), dti_to_epoch(epi_model.times), True
    )
    return runner, istate

In [ ]:
base_params = {
    "contact_rate": 15.0,
    "freq_dens_exponent": 1.0,
    "rel_sus_mtb_naive": 1.0,
    "rel_sus_contained": 0.3324848208129065,
    "rel_sus_cleared": 0.7101012998861186,
    "containment_age0": 4.4,
    "containment_age5": 4.4,
    "containment_age15": 2.0,
    "clearance_rate": 0.05643858771640714,
    "breakdown_rate": 0.5678500778834155,
    "progression_age0": 2.4,
    "progression_age5": 2.0,
    "progression_age15": 0.1,
    "progression_prop_infectious": 0.5,
    "increase_infect": 2.799432998282645,
    "decrease_infect": 1.0,
    "clinical_development": 2.280456422265371,
    "clinical_regression": 1.0,
    "self_recovery": 0.4,
    "seed_peak_time": 30.0,
    "seed_peak_rate": 0.01,
    "seed_duration": 10.0,
    "passive_detection_current": 3.1435769682295085,
    "passive_detection_shape": 0.12804084477668393,
    "passive_detection_inflection": 2006.359353581523 - start_time,
    "passive_detection_past": 2.237577299673147,
    "a_spread": 9.832801834382463,
    "bg_mixing": 0.029353135750061505,
    "pc_strength": 0.9853729910300592,
    "young_suscept": 0.5,
    "rx_duration": 0.5,
    "tsr": 0.7,
    "prop_neg_rx_death": 0.4,
    "rel_infectiousness_lowinf": 0.4,
    "rel_infectiousness_subclin": 0.5,
}

In [ ]:
runner, istate = get_runner(epi_model)
results = epi_model.run(base_params)

In [ ]:
results["compartments"].sumcats(compartment=disease_state.categories()).to_pandas_df().loc[1950:, :].plot.area()

In [ ]:
pd.DataFrame(results["flows"]["detection"].sum(to_dims="time").data).plot()

In [ ]:
# results["compartments"].sumcats(compartment=age_strat.categories()).to_pandas_df().loc[1950:, :].plot.area()

In [ ]:
LATENT_COMPS = ["incipient", "contained", "cleared"]
priors = {
    "contact_rate": dist.Uniform(0.005, 0.025),
    "recent_detection_rate": dist.Uniform(0.05, 0.5),
}
latent_date = 2019.0
latent_target = 0.43

notif_date = 2019.0
notif_target = 1e5


In [ ]:
def model():
    params = base_params | {k: numpyro.sample(k, v) for k, v in priors.items()}
    # If you get NaNs or Infs in your likelihood, max_steps might need to be increased
    results = epi_model.run(params, solver_kwargs={"max_steps": 2000})

    latent_vals = results["compartments"].query(compartment=disease_state[LATENT_COMPS], time=latent_date)
    latent_pop = latent_vals.sum(to_dims="time")
    total = results["compartments"].query(time=latent_date).sum(to_dims="time")
    latent_prop = latent_pop / (total + 1e-32)
    latent_ll = dist.Normal(latent_target, 0.1).log_prob(latent_prop.data[0])

    notif_val = results["flows"]["detection"].query(time=notif_date).sum()
    notif_ll = dist.Normal(notif_target, 2e5).log_prob(notif_val)

    ll = latent_ll + notif_ll

    numpyro.factor("ll", ll)


# Using SA here just to get a quick result, but have another look at NUTS once things are going more smoothly
kernel = infer.SA(model, adapt_state_size=4)  # infer.NUTS(model, max_tree_depth=5)
# warmup isn't really distinct from sampling here, effectively just acting as burn-in
mcmc = infer.MCMC(kernel, num_warmup=100, num_samples=100, num_chains=4)
mcmc.run(PRNGKey(0))

In [ ]:
idata = mcmc.get_samples(True)
az.summary(idata)
az.plot_trace(idata, compact=False)

In [ ]:
n_samples = 5

samples = az.from_dict(mcmc.get_samples(True))
n_chains = samples.posterior.dims["chain"]
n_draws = samples.posterior.dims["draw"]
chain_idx = np.random.randint(0, n_chains, size=n_samples)
draw_idx = np.random.randint(0, n_draws, size=n_samples)

latent_df = pd.DataFrame(index=epi_model.times)
notif_df = pd.DataFrame(index=epi_model.times)
for c, d in zip(chain_idx, draw_idx):
    sample_params = {name: values[c, d] for name, values in idata.items()}
    sample_results = epi_model.run(base_params | sample_params, solver_kwargs={"max_steps": 2000})
    
    latent = sample_results["compartments"].query(compartment=disease_state[LATENT_COMPS]).sum(to_dims="time").data
    total = sample_results["compartments"].sum(to_dims="time").data
    latent_df[f"{c}-{d}"] = latent / total

    notif_df[f"{c}-{d}"] = sample_results["flows"]["detection"].sum(to_dims="time").data


In [ ]:
fig = notif_df.plot()
fig.update_traces(line=dict(color="black", width=0.5))
fig.add_trace(go.Scatter(x=[notif_date], y=[notif_target]))

In [ ]:
fig = latent_df.plot()
fig.update_traces(line=dict(color="black", width=0.5))
fig.add_trace(go.Scatter(x=[latent_date], y=[latent_target]))